# Does DCUNet fail to *generalize*, or is it just weak?

Three checkpoints, all **F1 Pass A** — trained on the *same* drone-noise pool, so this
compares architectures, not recipes:

| model | checkpoint |
|---|---|
| DCUNet | `f1_dcunet_a` |
| Edge-BS-RoFormer | `f1_edge_bs_rof_a` |
| MP-SENet | `f1_mpsenet_a` |

Scored on three conditions that walk **one axis** — how far the noise is from what the
model trained on — with the *same 25 held-out LibriSpeech speakers* throughout, so
speech is controlled and only noise provenance changes:

| condition | noise |
|---|---|
| `seen` | the F1 Pass-A **training** pool itself — noise these checkpoints were fitted on |
| `unseen_rec` | held-out **recordings** of the same drone datasets |
| `unseen_drone` | **AVQ** — a quadrotor absent from training entirely |

The `seen` condition is the one no published valid set provides (they all hold noise
out by design), and it is what makes "fits in-distribution, fails out" visible rather
than merely asserted.

> Everything runs on **CPU**. Model loading dominates; keep `n_per_snr` small (2–5).
> The first run downloads three checkpoints from R2.

In [ ]:
%load_ext autoreload
%autoreload 2

import generalization_lib as gl

gl.CONDITIONS, list(gl.MODELS)

## 1. Run the probe

Raise `n_per_snr` for tighter means, or trim `snrs`/`models` to go faster.
Set `device="cuda"` if you have a GPU.

In [ ]:
df, samples = gl.run(
    n_per_snr=3,
    snrs=[-20, -15, -10, -5],
    device="cpu",
)
df.head()

## 2. The headline

Read the `Δ vs noisy` rows. The claim to check: **DCUNet's Δ collapses** as you move
left→right across conditions, while the other two hold up.

In [ ]:
print("eSTOI — intelligibility")
display(gl.summary_table(df, "estoi"))
print("\nSI-SDR (dB) — noise energy removed")
display(gl.summary_table(df, "si_sdr"))

In [ ]:
gl.plot_summary(df, "estoi");

In [ ]:
gl.plot_summary(df, "si_sdr");

### Why show both metrics

They come apart, and that *is* the failure mode. On unseen noise DCUNet keeps removing
noise **energy** (SI-SDR stays positive) while recovering essentially no
**intelligibility** (ΔeSTOI → 0). A model can look like it is denoising on SI-SDR alone
and be useless.

## 3. Listen to it

Same clip through every model, with spectrograms. Compare the **same index** across
conditions — that is where DCUNet's degradation is audible while the others stay
intelligible.

In [ ]:
gl.listen(samples, condition="seen", snr=-15, index=0);

In [ ]:
gl.listen(samples, condition="unseen_rec", snr=-15, index=0);

In [ ]:
gl.listen(samples, condition="unseen_drone", snr=-15, index=0);

---

# Part B — the actual "fits in-distribution, fails out" demo

**Part A above will not show that**, and it is worth understanding why before reading it.
The Pass-A checkpoints were trained on the *broad* drone pool, so their
"in-distribution" condition is itself broad and diverse. DCUNet does not fit it well
either — expect its ΔeSTOI to be ~0 or negative in **all** three conditions. That is
"weak everywhere", not "memorises then fails".

To see memorisation you need a **narrowly** trained model. `f2_dcunet_avq_heldout` was
trained on AVQ **session 1 only**, so:

| condition | for this checkpoint |
|---|---|
| `avq_seen` (`avq_ego_s1`) | noise it trained on |
| `avq_unseen` (`avq_ego_s2`) | same drone, different session, never seen |

The report measures this gap at **12.9 dB SI-SDR / 0.17 eSTOI** at −15 dB.

> There is **no equivalent narrowly-trained Edge-BS-RoFormer or MP-SENet checkpoint**,
> so the strict cross-architecture version of this experiment has not been run. What we
> do know is Part A: on held-out noise those two still improve intelligibility and
> DCUNet does not.

In [ ]:
df_b, samples_b = gl.run(
    n_per_snr=3,
    snrs=[-20, -15, -10, -5],
    models=["DCUNet (AVQ sess.1)"],
    conditions=["avq_seen", "avq_unseen"],
    duration_s=3.0,   # the F2 arms were trained on 3 s crops
    device="cpu",
)
display(gl.summary_table(df_b, "estoi"))
display(gl.summary_table(df_b, "si_sdr"))

In [ ]:
gl.plot_summary(df_b, "estoi");

### Hear the memorisation

Same model, same drone, same speakers — the only difference is whether it trained on
that recording session.

In [ ]:
gl.listen(samples_b, condition="avq_seen", snr=-15, index=0);

In [ ]:
gl.listen(samples_b, condition="avq_unseen", snr=-15, index=0);

## 4. Caveats worth keeping in view

- The **12.9 dB seen-vs-unseen gap** in the report was measured for DCUNet with a
  dedicated trained probe (`f2_dcunet_avq_heldout`, train on AVQ session 1, score both
  sessions). This notebook is the *cheap* version of that experiment for all three
  architectures using existing checkpoints — indicative, not the same controlled
  measurement.
- `seen` samples are drawn from the training *distribution*, not the exact training
  clips, so it understates true memorization somewhat.
- Small `n_per_snr` means noisy means. Raise it before quoting any number.
- Edge-BS-RoFormer runs with FlashAttention disabled on CPU; that changes speed, not
  output.